## 1. Data Cleaning


In [38]:
import pandas as pd

file_path = "automobile_prices_economics_2019_2023.csv"
df = pd.read_csv(file_path)

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (61, 6)


,Month/Year,New Price ($),Used Price ($),Inflation Rate (%),Interest Rate (%),Units Sold
0,19-Jan,"25,000","18,000",1.60%,2.40%,"28,500"
1,19-Feb,"25,000","17,900",1.50%,2.40%,"29,000"
2,19-Mar,"25,000","17,800",1.90%,2.40%,"31,500"
3,19-Apr,"25,000","17,700",2.00%,2.40%,"30,000"
4,19-May,"25,000","17,600",1.80%,2.40%,"32,000"


In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Month/Year          60 non-null     str  
 1   New Price ($)       60 non-null     str  
 2   Used Price ($)      60 non-null     str  
 3   Inflation Rate (%)  60 non-null     str  
 4   Interest Rate (%)   60 non-null     str  
 5   Units Sold          60 non-null     str  
dtypes: str(6)
memory usage: 3.0 KB


In [40]:
df[df.duplicated(keep=False)]

,Month/Year,New Price ($),Used Price ($),Inflation Rate (%),Interest Rate (%),Units Sold


In [41]:
# We find 1 row with null values
df[df.isna().any(axis=1)]
df = df.dropna(how='all')

In [42]:
# Convert to date Month/Year
df['Month/Year'] = pd.to_datetime(df['Month/Year'], format='%y-%b').dt.to_period('M')
df.head()

,Month/Year,New Price ($),Used Price ($),Inflation Rate (%),Interest Rate (%),Units Sold
0,2019-01,"25,000","18,000",1.60%,2.40%,"28,500"
1,2019-02,"25,000","17,900",1.50%,2.40%,"29,000"
2,2019-03,"25,000","17,800",1.90%,2.40%,"31,500"
3,2019-04,"25,000","17,700",2.00%,2.40%,"30,000"
4,2019-05,"25,000","17,600",1.80%,2.40%,"32,000"


In [43]:
non_numeric_chars = df['New Price ($)'].astype(str).str.findall(r'[^0-9.]').explode().unique()
print(f"Symbols in New Price ($):", non_numeric_chars)

non_numeric_chars = df['Used Price ($)'].astype(str).str.findall(r'[^0-9.]').explode().unique()
print(f"Symbols Used Price ($):", non_numeric_chars)

non_numeric_inflation = df['Inflation Rate (%)'].astype(str).str.findall(r'[^0-9.]').explode().unique()
print("Symbols in Inflation Rate (%):", non_numeric_inflation)

non_numeric_interest = df['Interest Rate (%)'].astype(str).str.findall(r'[^0-9.]').explode().unique()
print("Symbols in Interest Rate (%):", non_numeric_interest)

non_numeric_units = df['Units Sold'].astype(str).str.findall(r'[^0-9]').explode().unique()
print("Symbols in Units Sold:", non_numeric_units)

Symbols in New Price ($): <StringArray>
[',']
Length: 1, dtype: str
Symbols Used Price ($): <StringArray>
[',']
Length: 1, dtype: str
Symbols in Inflation Rate (%): <StringArray>
['%']
Length: 1, dtype: str
Symbols in Interest Rate (%): <StringArray>
['%']
Length: 1, dtype: str
Symbols in Units Sold: <StringArray>
[',']
Length: 1, dtype: str


In [44]:
# Remove symbols and convert New Price ($), Used Price ($), Inflation Rate (%) and Interest Rate (%) to integer and float and charachters to NaN
columns_to_clean = ['New Price ($)', 'Used Price ($)','Inflation Rate (%)', 'Interest Rate (%)', 'Units Sold']

for col in columns_to_clean:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(r'[^0-9.-]', '', regex=True) 
        .str.strip()
    )

for col in columns_to_clean:
    remaining_symbols = df[col].str.findall(r'[^0-9.-]').explode().unique()
    print(f"Remaining symbols in {col}:", remaining_symbols)

    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df[['New Price ($)', 'Used Price ($)','Inflation Rate (%)', 'Interest Rate (%)', 'Units Sold']].dtypes)
df.head()

Remaining symbols in New Price ($): [nan]
Remaining symbols in Used Price ($): [nan]
Remaining symbols in Inflation Rate (%): [nan]
Remaining symbols in Interest Rate (%): [nan]
Remaining symbols in Units Sold: [nan]
New Price ($)           int64
Used Price ($)          int64
Inflation Rate (%)    float64
Interest Rate (%)     float64
Units Sold              int64
dtype: object


,Month/Year,New Price ($),Used Price ($),Inflation Rate (%),Interest Rate (%),Units Sold
0,2019-01,25000,18000,1.6,2.4,28500
1,2019-02,25000,17900,1.5,2.4,29000
2,2019-03,25000,17800,1.9,2.4,31500
3,2019-04,25000,17700,2.0,2.4,30000
4,2019-05,25000,17600,1.8,2.4,32000


In [45]:
numeric_cols = ['New Price ($)', 'Used Price ($)', 'Inflation Rate (%)', 'Interest Rate (%)', 'Units Sold']

failed_rows = df[df[numeric_cols].isna().any(axis=1)]

if len(failed_rows) != 0:
    print(f"Number of rows with conversion errors: {len(failed_rows)}")
    failed_rows

In [46]:
df.describe()

,New Price ($),Used Price ($),Inflation Rate (%),Interest Rate (%),Units Sold
count,60.000000,60.000000,60.000000,60.000000,60.000000
mean,25543.333333,15580.000000,4.125000,2.075833,42700.000000
std,533.732195,1210.658877,2.660867,1.879923,15888.435193
min,25000.000000,13800.000000,0.100000,0.250000,18000.000000
25%,25200.000000,14600.000000,1.675000,0.250000,29375.000000
50%,25400.000000,15350.000000,4.100000,2.075000,37250.000000
75%,25700.000000,16525.000000,6.250000,3.125000,55250.000000
max,27000.000000,18000.000000,9.100000,5.750000,75000.000000


1. We didn't find any duplicate rows and we removed the single row that contained missing values.
2. We converted the `Month/Year` column from string into a date.
3. We checked for special characters such as dollar signs, percentage signs, and commas in the remaining columns.
4. We removed all non-numeric symbols using regular expressions.
5. We converted `New Price ($)`, `Used Price ($)`, and `Units Sold` into integers.
6. We converted `Inflation Rate (%)` and `Interest Rate (%)` into float64.

## 2. Exploratory Analysis
